# ACTIVE-TB Step E: sealed evaluation

Verifies frozen suite hashes, then runs Golden RTL and Hidden Bank A through the frozen CorrectBench checkers with Icarus Verilog. No search or generation code is present in this dataset.


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, zipfile
INPUT=Path('/kaggle/input'); WORK=Path('/kaggle/working')
m=list(INPUT.rglob('active_tb_distill_v1/SEALED_BOUNDARY.json'))
if len(m)==1: SRC=m[0].parent
else:
 a=list(INPUT.rglob('active_tb_distill_v1.zip')); assert len(a)==1,a
 d=WORK/'attached'
 with zipfile.ZipFile(a[0]) as zf: zf.extractall(d)
 SRC=d/'active_tb_distill_v1'
sz=SRC/'suites'
sd=SRC/'suites.zip'
if not sz.is_dir() and sd.exists():
 sz=WORK/'suites'
 with zipfile.ZipFile(sd) as zf: zf.extractall(sz)
b=json.loads((SRC/'SEALED_BOUNDARY.json').read_text())
print('boundary:',b)
print('suite files:',len(list(sz.glob('*.v'))))
subprocess.run(['apt-get','update','-qq'],check=True)
subprocess.run(['apt-get','install','-y','-qq','iverilog'],check=True)
print('iverilog ready')


In [ ]:
OUT=WORK/'distill_results'; OUT.mkdir(exist_ok=True)
cmd=[sys.executable,str(SRC/'sealed_evaluator.py'),
 '--suites',str(sz),
 '--manifest',str(SRC/'suite_manifest.json'),
 '--checkers',str(SRC/'checkers.jsonl'),
 '--golden',str(SRC/'golden.jsonl'),
 '--mutants',str(SRC/'hidden_bank_a.jsonl'),
 '--out',str(OUT/'sealed_records.csv')]
r=subprocess.run(cmd,text=True,capture_output=True)
print(r.stdout[-4000:])
if r.returncode: print(r.stderr[-4000:]); raise SystemExit('evaluator failed')
shutil.copy2(SRC/'suite_manifest.json',OUT/'suite_manifest.json')
shutil.make_archive(str(WORK/'distill_results'),'zip',OUT)
print('packaged')
